### weight map

In [1]:
import torch, math

state_dict = torch.load(
    "pytorch_model.bin", map_location="cpu", weights_only=True, mmap=True
)

In [2]:
import json

with open("modelWeighStructure.json", "w") as f:
    model_structure = {
        layer_name: state_dict[layer_name].shape for layer_name in state_dict.keys()
    }

    json.dump(model_structure, f, indent=4)

In [3]:
for layer, shape in json.load(open("modelWeighStructure.json")).items():
    if 'model.decoder' in layer:
        print(f"Name: {layer} | Shape: {shape}")

Name: model.decoder.embed_tokens.weight | Shape: [256206, 1024]
Name: model.decoder.layers.0.self_attn.k_proj.weight | Shape: [1024, 1024]
Name: model.decoder.layers.0.self_attn.k_proj.bias | Shape: [1024]
Name: model.decoder.layers.0.self_attn.v_proj.weight | Shape: [1024, 1024]
Name: model.decoder.layers.0.self_attn.v_proj.bias | Shape: [1024]
Name: model.decoder.layers.0.self_attn.q_proj.weight | Shape: [1024, 1024]
Name: model.decoder.layers.0.self_attn.q_proj.bias | Shape: [1024]
Name: model.decoder.layers.0.self_attn.out_proj.weight | Shape: [1024, 1024]
Name: model.decoder.layers.0.self_attn.out_proj.bias | Shape: [1024]
Name: model.decoder.layers.0.self_attn_layer_norm.weight | Shape: [1024]
Name: model.decoder.layers.0.self_attn_layer_norm.bias | Shape: [1024]
Name: model.decoder.layers.0.encoder_attn.k_proj.weight | Shape: [1024, 1024]
Name: model.decoder.layers.0.encoder_attn.k_proj.bias | Shape: [1024]
Name: model.decoder.layers.0.encoder_attn.v_proj.weight | Shape: [1024, 

## Rebuilding model
### Encoder

In [4]:
class MultiHeadAtention(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, state_dict: dict):
        super().__init__()

        self.d_model = config["d_model"]
        self.num_attn_head = config["encoder_attention_heads"]
        self.attn_dim = self.d_model // self.num_attn_head

        self.k_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.q_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.v_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.out_proj = torch.nn.Linear(self.d_model, self.d_model)

        # weight
        self.k_proj.weight.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.k_proj.weight"]
        )
        self.q_proj.weight.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.q_proj.weight"]
        )
        self.v_proj.weight.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.v_proj.weight"]
        )
        self.out_proj.weight.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.out_proj.weight"]
        )

        # bias
        self.k_proj.bias.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.k_proj.bias"]
        )
        self.q_proj.bias.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.q_proj.bias"]
        )
        self.v_proj.bias.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.v_proj.bias"]
        )
        self.out_proj.bias.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn.out_proj.bias"]
        )

    @torch.no_grad()
    def forward(self, x):
        B, S, _ = x.shape

        k = self.k_proj(x).view(B, S, self.num_attn_head, self.attn_dim).transpose(1, 2)
        q = self.q_proj(x).view(B, S, self.num_attn_head, self.attn_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.num_attn_head, self.attn_dim).transpose(1, 2)

        attn_scores = k @ q.transpose(-2, -1) * (self.attn_dim**-0.5)
        attn_scores = torch.nn.functional.softmax(attn_scores, dim=-1)

        out = attn_scores @ v
        out = out.transpose(1, 2).contiguous().view(B, S, self.d_model)
        out = self.out_proj(out)

        return out

In [5]:
class EncoderFeedForwardNetwork(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, state_dict: dict):
        super().__init__()

        self.d_model = config["d_model"]
        self.ffn_dim = config["encoder_ffn_dim"]

        self.fc1 = torch.nn.Linear(in_features=self.d_model, out_features=self.ffn_dim)
        self.act = torch.nn.ReLU()
        self.fc2 = torch.nn.Linear(in_features=self.ffn_dim, out_features=self.d_model)

        # weight
        self.fc1.weight.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.fc1.weight"]
        )
        self.fc2.weight.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.fc2.weight"]
        )

        # bias
        self.fc1.bias.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.fc1.bias"]
        )
        self.fc2.bias.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.fc2.bias"]
        )

    @torch.no_grad()
    def forward(self, x):

        return self.fc2(self.act(self.fc1(x)))

In [6]:
class EncoderLayer(torch.nn.Module):

    def __init__(self, config: dict, layer_no: int, state_dict: dict):
        super().__init__()
        self.d_model = config["d_model"]

        self.self_attn_layer_norm = torch.nn.LayerNorm(self.d_model)
        self.self_attn = MultiHeadAtention(config, layer_no, state_dict)
        self.ffn = EncoderFeedForwardNetwork(config, layer_no, state_dict)
        self.final_layer_norm = torch.nn.LayerNorm(self.d_model)

        # weight
        self.self_attn_layer_norm.weight.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn_layer_norm.weight"]
        )
        self.final_layer_norm.weight.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.final_layer_norm.weight"]
        )

        # bias
        self.self_attn_layer_norm.bias.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.self_attn_layer_norm.bias"]
        )
        self.final_layer_norm.bias.data.copy_(
            state_dict[f"model.encoder.layers.{layer_no}.final_layer_norm.bias"]
        )

    @torch.no_grad()
    def forward(self, x):

        residual = x
        x = residual + self.self_attn(self.self_attn_layer_norm(x))

        residual = x
        return residual + self.ffn(self.final_layer_norm(x))

In [7]:
class SinusoidalPositionalEmbedding(torch.nn.Module):

    def __init__(
        self,
        num_positions: int,
        embedding_dim: int,
        padding_idx: int | None = None,
    ):
        super().__init__()

        self.offset = 2
        self.num_positions = num_positions
        self.embedding_dim = embedding_dim
        self.padding_idx = padding_idx

        self.register_buffer(
            "weights",
            self.get_embedding(
                num_positions + self.offset,
                embedding_dim,
                padding_idx,
            ),
            persistent=False,
        )

    @staticmethod
    def get_embedding(
        num_embeddings: int,
        embedding_dim: int,
        padding_idx: int | None = None,
    ):

        half_dim = embedding_dim // 2

        exponent = math.log(10000.0) / (half_dim - 1)

        frequencies = torch.exp(
            torch.arange(half_dim, dtype=torch.float32)
            * -exponent
        )

        positions = torch.arange(
            num_embeddings, dtype=torch.float32
        ).unsqueeze(1)

        angles = positions * frequencies.unsqueeze(0)

        embeddings = torch.cat(
            [torch.sin(angles), torch.cos(angles)],
            dim=1,
        )

        if embedding_dim % 2 == 1:
            embeddings = torch.cat(
                [
                    embeddings,
                    torch.zeros(num_embeddings, 1),
                ],
                dim=1,
            )

        if padding_idx is not None:
            embeddings[padding_idx, :] = 0

        return embeddings

    @staticmethod
    def create_position_ids_from_input_ids(
        input_ids: torch.Tensor,
        padding_idx: int,
        past_key_values_length: int = 0,
    ):

        mask = input_ids.ne(padding_idx).int()

        incremental_indices = (
            torch.cumsum(mask, dim=1)
            + past_key_values_length
        )

        return (
            incremental_indices * mask
        ).long() + padding_idx

    @torch.no_grad()
    def forward(
        self,
        input_ids: torch.Tensor,
        past_key_values_length: int = 0,
    ):

        position_ids = self.create_position_ids_from_input_ids(
            input_ids,
            self.padding_idx,
            past_key_values_length,
        )

        max_position = position_ids.max().item()

        if max_position >= self.weights.size(0):
            raise ValueError(
                f"Position {max_position} exceeds positional "
                f"embedding table size {self.weights.size(0)}"
            )

        return self.weights[position_ids]

In [8]:
class NllbEncoder(torch.nn.Module):

    def __init__(self, config: dict, state_dict: dict):
        super().__init__()

        self.token_embed = torch.nn.Embedding(
            num_embeddings=config["vocab_size"],
            embedding_dim=config["d_model"],
            padding_idx=config["pad_token_id"],
        )

        self.embed_scale = (
            math.sqrt(config["d_model"]) if config["scale_embedding"] else 1.0
        )

        self.embed_positions = SinusoidalPositionalEmbedding(
            num_positions=config["max_position_embeddings"],
            embedding_dim=config["d_model"],
            padding_idx=config["pad_token_id"],
        )

        self.layers = torch.nn.ModuleList(
            EncoderLayer(config, i, state_dict) for i in range(config["encoder_layers"])
        )

        self.final_layer_norm = torch.nn.LayerNorm(config["d_model"])

        # weight
        self.token_embed.weight.data.copy_(
            state_dict["model.encoder.embed_tokens.weight"]
        )
        self.final_layer_norm.weight.data.copy_(
            state_dict["model.encoder.layer_norm.weight"]
        )

        # bias
        self.final_layer_norm.bias.data.copy_(
            state_dict["model.encoder.layer_norm.bias"]
        )

    @torch.no_grad()
    def forward(self, x):

        positions = self.embed_positions(x)

        x = self.token_embed(x) * self.embed_scale
        x = x + positions.to(dtype=x.dtype)

        for layer in self.layers:
            x = layer(x)

        return self.final_layer_norm(x)

In [28]:
class MaskedMultiHeadAttention(torch.nn.Module):
    def __init__(self, config: dict, layer_no: int, state_dict: dict):
        super().__init__()

        self.d_model = config["d_model"]
        self.num_attn_head = config["decoder_attention_heads"]
        self.attn_dim = self.d_model // self.num_attn_head

        self.k_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.q_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.v_proj = torch.nn.Linear(self.d_model, self.d_model)
        self.out_proj = torch.nn.Linear(self.d_model, self.d_model)

        # weight
        self.k_proj.weight.data.copy_(
            state_dict[f"model.decoder.layers.{layer_no}.self_attn.k_proj.weight"]
        )
        self.q_proj.weight.data.copy_(
            state_dict[f"model.decoder.layers.{layer_no}.self_attn.q_proj.weight"]
        )
        self.v_proj.weight.data.copy_(
            state_dict[f"model.decoder.layers.{layer_no}.self_attn.v_proj.weight"]
        )
        self.out_proj.weight.data.copy_(
            state_dict[f"model.decoder.layers.{layer_no}.self_attn.out_proj.weight"]
        )

        # bias
        self.k_proj.bias.data.copy_(
            state_dict[f"model.decoder.layers.{layer_no}.self_attn.k_proj.bias"]
        )
        self.q_proj.bias.data.copy_(
            state_dict[f"model.decoder.layers.{layer_no}.self_attn.q_proj.bias"]
        )
        self.v_proj.bias.data.copy_(
            state_dict[f"model.decoder.layers.{layer_no}.self_attn.v_proj.bias"]
        )
        self.out_proj.biad.data.copy_(
            state_dict[f"model.decoder.layers.{layer_no}.self_attn.out_proj.bias"]
        )

    def forward(self, x):

        B, S, _ = x.shape

        k = self.k_proj(x).view(B, S, self.num_attn_head, self.attn_dim).transpose(1, 2)
        q = self.q_proj(x).view(B, S, self.num_attn_head, self.attn_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.num_attn_head, self.attn_dim).transpose(1, 2)

        attn_scores = k @ q.transpose(-2, -1) * (self.attn_dim**-0.5)

        if S > 1:
            mask = torch.triu(torch.ones(S, S), diagonal=1).bool()
            attn_scores = attn_scores.masked_fill(mask, float("-inf"))

        attn_scores = torch.nn.functional.softmax(attn_scores, dim=-1)

        out = attn_scores @ v
        out = out.transpose(1, 2).contiguous().view(B, S, self.d_model)

        return self.out_proj(out)

In [19]:
model_config = json.load(open("config.json"))
device = torch.device('cuda')

model_input = torch.randint(0, model_config["vocab_size"], (2, 12)).to(device)

encoder = NllbEncoder(model_config, state_dict).to(device)
encoder_output = encoder(model_input)
encoder_output.shape

torch.Size([2, 12, 1024])

In [14]:
trainable_params = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
print(
    f"Total Trainable Parameters: {trainable_params:,} | {trainable_params * 4 //  1024**2} MB"
)

Total Trainable Parameters: 766,091,264 | 2922 MB
